# Feature Selection 
This notebook contains the code for feature selection using RF importance and mutual information. Comparing all features, raw only, and engineered only

### Imports

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from src.splits import load_split
from src.evaluate import evaluate, metrics_at_threshold, threshold_table, summarize, log_result, THRESHOLDS

In [ ]:
X_train, X_test, y_train, y_test = load_split("binary")
del X_test, y_test
print(list(X_train.columns))
print(X_train.head())

## Feature rankings

In [ ]:
ENGINEERED = ["Temp diff [K]", "Power [W]", "Strain [min Nm]"]
RAW = [c for c in X_train.columns if c not in ENGINEERED]

In [ ]:
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif

cols = list(X_train.columns)
is_discrete = np.array([c.startswith("Type_") for c in cols])

rf = RandomForestClassifier(n_estimators=500, class_weight="balanced", random_state=42, n_jobs=-1).fit(X_train, y_train)
ranking = pd.DataFrame({
    "rf_importance": rf.feature_importances_,
    "mutual_info": mutual_info_classif(X_train, y_train, discrete_features=is_discrete, random_state=42),
}, index=cols).sort_values("rf_importance", ascending=False)
ranking

 **Binary rankings**

- By RF importance, rotational speed, torque and power lead (0.20, 0.19, 0.18), followed by tool wear (0.14), strain (0.12) and temperature difference (0.09).
- Power is computed from torque × rotational speed, so these three share information and the importance is split between them. A low rank therefore does not prove a feature is useless; the cross-validated comparison below is the real test.
- RF importance and mutual information agree on the top group, except tool wear: 4th by RF importance but low by mutual information (0.012). This is consistent with tool wear acting through interactions (e.g. in strain = wear × torque), which a univariate measure cannot see.
- Rankings are computed on the full training set and impurity importance favours continuous features, so they are used to choose candidate sets only.

## Binary feature set comparison

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_selection import SelectKBest, SelectFromModel
from functools import partial

def make_rf():
    return RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42, n_jobs=-1)

mi_score = partial(mutual_info_classif, discrete_features=is_discrete, random_state=42)

no_type = [c for c in cols if not c.startswith("Type_")]
no_temps = [c for c in cols if c not in ("Air temperature [K]", "Process temperature [K]")]
best6 = [c for c in no_type if c in no_temps]

FEATURE_SETS = {
    "all features": (cols, None),
    "raw only": (RAW, None),
    "engineered only": (ENGINEERED, None),
    "all but type": (no_type, None),
    "all minus air/process temp": (no_temps, None),
    "best 6 features": (best6, None),
}

In [ ]:
rows = []

for name, (use_cols, selector) in FEATURE_SETS.items():
    model = make_pipeline(selector, make_rf()) if selector else make_rf()

    results, probas = evaluate(model, X_train[use_cols], y_train, threshold=0.5)
    s = summarize(results)

    tt = threshold_table(probas, THRESHOLDS)
    best = tt.loc[tt["f1_mean"].idxmax()]

    rows.append({"feature_set": name, "n_features": len(use_cols),
                 "f1": s["f1_mean"], "f1_std": s["f1_std"],
                 "recall": s["recall_mean"], "precision": s["precision_mean"],
                 "fn": s["fn_mean"], "fp": s["fp_mean"],
                 "best_thr": best["threshold"], "f1_best_thr": best["f1_mean"]})

    # Log results to file
    log_result("RF", "binary", {"feature_set": name, "n_estimators": 300,
                                "class_weight": "balanced"}, 0.5, s)

In [ ]:
table = pd.DataFrame(rows)
table.to_csv(pathlib.Path.cwd().parent / "results" / "feature_selection_binary.csv", index=False)
table

**Binary feature sets**

Fixed RF (300 trees, balanced class weights), 5-fold × 3 repeats, validation only.

- Both feature groups are needed: Raw only and engineered only both fall about 0.17–0.19 F1 below the full set, far more than the fold std. Three engineered features beat eight raw ones, but neither group is enough alone.
- Air and process temperature can be dropped.
- Type seems to help, mainly by reducing false alarms. Removing it lowers F1 by about 0.026 (about one std). Recall stays the same (0.81), but false alarms rise from 4.7 to 7.5 per fold and precision falls from 0.90 to 0.85.

## Multiclass check

In [ ]:
X_m, X_te, y_m, y_te = load_split("multi")
del X_te, y_te

cols_m = list(X_m.columns)
is_discrete_m = np.array([c.startswith("Type_") for c in cols_m])

rf_m = RandomForestClassifier(n_estimators=500, class_weight="balanced",
                              random_state=42, n_jobs=-1).fit(X_m, y_m)
ranking_m = pd.DataFrame({
    "rf_importance": rf_m.feature_importances_,
    "mutual_info": mutual_info_classif(X_m, y_m, discrete_features=is_discrete_m,
                                       random_state=42),
}, index=cols_m).sort_values("rf_importance", ascending=False)
ranking_m

In [ ]:
print("MultiClass",ranking_m)
print("\n","Binary", ranking)

**Multiclass rankings vs binary**

- The ranking changes: tool wear is 1st and strain 2nd, followed by power, torque and rotational speed. In the binary ranking tool wear was 4th and rotational speed 1st.
- This is expected: the multiclass target separates failure modes, tool wear and overstrain depend on wear and strain, heat and power failures on temperature difference, speed and power, while the binary target merges them.
- Power and strain are in the top five for both tasks, so the engineered features are useful in both settings.
- Air/process temperature and Type are again at the bottom, though Type_L is somewhat higher here (0.013 vs 0.004).

In [ ]:
from sklearn.model_selection import cross_validate
from src.splits import get_cv

cv = get_cv(n_repeats=3)
for name, (use_cols, selector) in FEATURE_SETS.items():
    model = make_pipeline(selector, make_rf()) if selector else make_rf()
    out = cross_validate(model, X_m[use_cols], y_m, cv=cv,
                         scoring="f1_macro", n_jobs=1)
    print(name, out["test_score"].mean().round(3), "±", out["test_score"].std().round(3))

**Multiclass feature sets**

Macro F1 (5 classes), same RF, 5-fold × 3 repeats, validation only. These rows use different data from the binary table (task-specific row removal), so scores are not comparable across tasks.

- **The same pattern as binary:** the full set beats raw only and engineered only by about 0.11–0.14, which is well outside the std.
- Dropping air/process temperature is again harmless(0.775 vs 0.768, a tie).
- Type matters little here. All four sets with the main features are within about 0.012 of each other, which is within noise. The Type effect seen in the binary task is not confirmed for multiclass.
- Macro F1 includes the tool wear class (TWF), which is expected to score near zero, so it limits every set equally.

## Decisions

**Selected feature set**
Rotational speed, torque, tool wear, temperature difference, power, strain, Type_L, Type_M, Type_H.
(Dropped: air temperature and process temperature.)

**Why**
- It ties the best score in both tasks (binary F1 0.855 vs 0.854 for all features; multiclass macro F1 0.775 vs 0.768), so we choose the simpler set.
- Fewer redundant inputs should also help the MLP.
- Type is kept because removing it raised false alarms in the binary task (4.7 → 7.5 per fold) and costs nothing to keep.
- The engineered features are kept in every model, since neither group works alone.

In [ ]:
import json
SELECTED = [c for c in cols if c not in ("Air temperature [K]", "Process temperature [K]")]
json.dump(SELECTED, open(pathlib.Path.cwd().parent / "results" / "selected_features.json", "w"), indent=2)
print(SELECTED)